In [1]:
from lenskit.algorithms.als import BiasedMF
import pandas as pd

import utils

c:\Users\sebas\.conda\envs\dtu-ct\lib\site-packages\llvmlite\binding\ffi.py:175: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


# CF Example
This notebook shows how the `BiasedMF` model from `LensKit` can be used to generate predicted scores using the matrix factorization technique of collaborative filtering (CF).

In [2]:
# test data to use in this example
test_data = pd.DataFrame({"user": [1,1,1,1,2,2,2,2,3,3,3,3],
                          "item": [1,4,6,7,2,8,4,9,3,5,6,7],
                          "rating": [0.4,0.3,0.2,0.2,0.4,0.5,0.9,0.1,0.1,0.4,0.3,0.3]})

In [ ]:
# lists of unique users and items
users = [1,2,3]
items = [1,2,3,4,5,6,7,8,9]

Using `predict_for_user` to generate predicted scores per user in a for loop (slow method):

In [ ]:
scores_dict = {}

# initializing the BiasedMF model
mf = BiasedMF(features=100, 
            iterations=10, 
            reg=0.1, 
            damping=5, 
            bias=True,
            rng_spec=42)

# fitting the model
mf.fit(test_data)

# generating scores for each user and saving to the scores_dict
for user in users:
    scores = list(mf.predict_for_user(user, items))
    scores_dict[user] = scores

print(scores_dict)

Numba is using threading layer omp - consider TBB
found 1 potential runtime problems - see https://boi.st/lkpy-perf


{1: [0.3229975009805931, 0.3186483487262771, 0.26875729002813353, 0.3440786771059767, 0.31812290472918914, 0.2799398355681748, 0.2799398355681748, 0.32528401904711157, 0.2987413377610054], 2: [0.37901326643266336, 0.40112873288474304, 0.35075410849277394, 0.6617976103155352, 0.4037323530648221, 0.37943051161754837, 0.37943051161754837, 0.4688059275100309, 0.19809714902293174], 3: [0.3307766001600723, 0.3309608129840696, 0.2787512463581338, 0.3965548445557614, 0.33182237150111865, 0.29519429974728917, 0.29519429974728917, 0.3480173300554078, 0.27979126177047975]}


Computing the matrix product UV and adding the bias terms to obtain the predicted ratings in a more efficient way:

In [ ]:
# U: [n_users × k]
U = mf.user_features_

# V: [n_items × k]
V = mf.item_features_

# biases:
ub = mf.bias.user_offsets_.reindex(mf.user_index_).to_numpy()
ib = mf.bias.item_offsets_.reindex(mf.item_index_).to_numpy()
mu = mf.bias.mean_
ub = ub.reshape(-1, 1)   # shape [n_users, 1]
ib = ib.reshape(1, -1)   # shape [1, n_items]

# full prediction matrix: [n_users × n_items]
pred_matrix = U @ V.T + ub + ib + mu
print(pred_matrix)

[[0.3229975  0.31864835 0.26875729 0.34407868 0.3181229  0.27993984
  0.27993984 0.32528402 0.29874134]
 [0.37901327 0.40112873 0.35075411 0.66179761 0.40373235 0.37943051
  0.37943051 0.46880593 0.19809715]
 [0.3307766  0.33096081 0.27875125 0.39655484 0.33182237 0.2951943
  0.2951943  0.34801733 0.27979126]]
